# Joko case study - statistical analysis

Two kinds of data, never mixed:
- **PUBLIC DATA**: cashback rates collected on two French cashback sites (iGraal, Widilo), App Store / Firefox reviews.
- **SIMULATED**: an invented user base (`src/simulate.py`, seed 42) used only to show the metrics and the retention model.

Sections
1. Same merchant, two platforms: how different are the cashback rates? (paired test, bootstrap CI)
2. Ratings by period and app version (with counts)
3. 90-day retention model (logistic regression, time hold-out, AUC, calibration)
4. Does the AI assistant increase purchases? Naive vs adjusted comparison (selection bias)

In [1]:
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "data" / "output"
rng = np.random.default_rng(42)

## 1. Same merchant, two platforms (PUBLIC DATA)

`sql/02_rate_comparison.sql` pairs every merchant that shows a **percentage** rate on both Widilo and iGraal on the same day.
- The gap is `Widilo rate - iGraal rate`, in percentage points.
- **Bootstrap**: resample the merchants 10,000 times with replacement, recompute the median gap each time; the 2.5th and 97.5th percentiles give a 95% confidence interval. No normality assumption.
- **Wilcoxon signed-rank test**: the paired non-parametric test; H0 = the gaps are symmetric around 0 (no systematic difference). Ties (gap = 0) are dropped by the test, as is standard.

In [2]:
pairs = pd.read_csv(OUT / "02_rate_comparison.csv")
gap = pairs["gap_widilo_minus_igraal_pts"].to_numpy()

boot = np.array([np.median(rng.choice(gap, size=len(gap), replace=True)) for _ in range(10_000)])
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
w = stats.wilcoxon(gap[gap != 0])

summary_rates = {
    "merchants_compared": int(len(gap)),
    "median_gap_pts": float(np.median(gap)),
    "median_gap_ci95": [float(ci_low), float(ci_high)],
    "mean_gap_pts": round(float(gap.mean()), 2),
    "share_widilo_higher": round(float((gap > 0).mean()), 3),
    "share_tie": round(float((gap == 0).mean()), 3),
    "share_igraal_higher": round(float((gap < 0).mean()), 3),
    "wilcoxon_p_value": float(w.pvalue),
    "median_relative_gap_pct": round(float(np.median((pairs.widilo_pct - pairs.igraal_pct) / pairs.igraal_pct) * 100), 1),
}
summary_rates

{'merchants_compared': 316,
 'median_gap_pts': 1.0,
 'median_gap_ci95': [0.75, 1.0],
 'mean_gap_pts': 1.04,
 'share_widilo_higher': 0.759,
 'share_tie': 0.051,
 'share_igraal_higher': 0.19,
 'wilcoxon_p_value': 4.861521682856109e-18,
 'median_relative_gap_pct': 25.0}

In [3]:
# Same test by category (only categories with at least 15 merchants compared)
rows = []
for cat, g in pairs.groupby("category"):
    if len(g) < 15:
        continue
    d = g["gap_widilo_minus_igraal_pts"].to_numpy()
    b = np.array([np.median(rng.choice(d, size=len(d), replace=True)) for _ in range(5_000)])
    rows.append({"category": cat, "n": len(d), "median_gap_pts": np.median(d),
                 "ci_low": np.percentile(b, 2.5), "ci_high": np.percentile(b, 97.5),
                 "share_widilo_higher": (d > 0).mean()})
by_cat = pd.DataFrame(rows).sort_values("median_gap_pts", ascending=False)
by_cat.to_csv(OUT / "nb_rate_gap_by_category.csv", index=False)
by_cat

,category,n,median_gap_pts,ci_low,ci_high,share_widilo_higher
1,B2B,15,1.000,0.500000,1.34,0.800000
3,Culture & leisure,15,1.000,0.100000,2.27,0.800000
2,Beauty & health,39,1.000,0.840000,1.70,0.871795
5,Fashion,50,1.000,0.500000,1.29,0.760000
4,Electronics,34,0.875,0.498125,1.00,0.735294
6,Home & garden,37,0.750,0.500000,2.00,0.864865
0,Auto,16,0.500,-0.500000,1.50,0.687500
7,Sports,25,0.500,-0.500000,2.50,0.560000
8,Travel,31,0.500,0.050000,1.00,0.677419


**Reading.** A positive gap means Widilo pays more. Widilo's category pages show its most visible merchants, which are more often boosted,
so the comparison is "Widilo's showcased offers vs iGraal's offer for the same merchants": it overstates the challenger's typical premium.
The robust conclusion is the **size of the dispersion**: for the same merchant, on the same day, French platforms do not pay the same rate.

## 2. Ratings by period and version (PUBLIC DATA)

Reviews come from Joko's public App Store pages (today and archived copies, 2019-2026) and from the Firefox add-on API.
They are a small, non-random sample: the App Store shows a handful of "most useful" reviews per page. Averages on fewer than 10 reviews are not interpreted.

In [4]:
rev_path = ROOT / "data" / "public" / ("fct_app_reviews_themed.csv" if (ROOT / "data/public/fct_app_reviews_themed.csv").exists() else "fct_app_reviews.csv")
rev = pd.read_csv(rev_path, parse_dates=["date"], keep_default_na=False, na_values=[""])
rev["year"] = rev["date"].dt.year
by_year = rev.groupby(["store", "year"]).agg(reviews=("rating", "size"), avg_rating=("rating", "mean"),
                                              share_1_2_stars=("rating", lambda s: (s <= 2).mean())).round(2)
by_year["interpretable"] = by_year["reviews"] >= 10
by_year.to_csv(OUT / "nb_reviews_by_year.csv")
by_year

reviews  avg_rating  share_1_2_stars  interpretable
store             year                                                     
app_store         2019       12        4.33             0.17           True
                  2020        3        5.00             0.00          False
                  2021        2        5.00             0.00          False
                  2022        3        5.00             0.00          False
                  2023        8        4.12             0.12          False
                  2024       10        4.90             0.00           True
                  2025       12        4.67             0.08           True
                  2026       21        4.29             0.14           True
firefox_extension 2021        3        4.67             0.00          False
                  2022        6        4.50             0.00          False
                  2023       20        4.55             0.00           True
                  2024       17        4.35             0.12           True
                  2025        4        4.00             0.25          False
                  2026        2        5.00             0.00          False

In [5]:
by_version = (rev[rev["store"] == "app_store"].dropna(subset=["major_version"])
              .groupby("major_version").agg(reviews=("rating", "size"), avg_rating=("rating", "mean")).round(2))
by_version["interpretable"] = by_version["reviews"] >= 10
by_version

,reviews,avg_rating,interpretable
major_version,,,
2.0,10,4.90,True
4.0,3,5.00,False
5.0,22,4.14,True


## 3. Retention model (SIMULATED)

**Question**: from what a new user does in their **first 14 days**, can we predict whether they are still active at **day 90**
(at least one session between day 61 and day 90)?

- Features (first 14 days): sessions, searches, merchant clicks, assistant queries, purchases, plus platform, acquisition channel and country.
- **Time hold-out**: train on users who signed up May 2025 - January 2026, test on February - July 2026 (later users, never seen in training). This mimics using the model on next month's sign-ups.
- Model: logistic regression on standardised features (coefficients comparable between features).
- Metrics: **AUC** (probability that a retained user gets a higher score than a churned one; 0.5 = random) and **calibration** (do users scored 30% really retain 30% of the time?).

In [6]:
con = duckdb.connect(str(ROOT / "data" / "simulated" / "joko_sim.duckdb"), read_only=True)
features = con.execute('''
    WITH first14 AS (
        SELECT e.user_id,
               sum(event_count) FILTER (WHERE event_type = 'session')         AS sessions_14d,
               sum(event_count) FILTER (WHERE event_type = 'search')          AS searches_14d,
               sum(event_count) FILTER (WHERE event_type = 'merchant_click')  AS clicks_14d,
               sum(event_count) FILTER (WHERE event_type = 'assistant_query') AS assistant_queries_14d
        FROM events e JOIN users u USING (user_id)
        WHERE e.event_date < u.signup_date + INTERVAL 14 DAY
        GROUP BY e.user_id
    ),
    purch14 AS (
        SELECT p.user_id, count(*) AS purchases_14d
        FROM purchases p JOIN users u USING (user_id)
        WHERE p.purchase_date < u.signup_date + INTERVAL 14 DAY
        GROUP BY p.user_id
    )
    SELECT m.user_id, m.signup_date, m.country, m.acquisition_channel, m.platform, m.retained_d90,
           coalesce(f.sessions_14d, 0) AS sessions_14d, coalesce(f.searches_14d, 0) AS searches_14d,
           coalesce(f.clicks_14d, 0) AS clicks_14d, coalesce(f.assistant_queries_14d, 0) AS assistant_queries_14d,
           coalesce(p.purchases_14d, 0) AS purchases_14d
    FROM v_user_metrics m
    LEFT JOIN first14 f USING (user_id)
    LEFT JOIN purch14 p USING (user_id)
    WHERE m.retained_d90 IS NOT NULL
''').fetchdf()
features["any_purchase_14d"] = (features["purchases_14d"] > 0).astype(int)
features["used_assistant_14d"] = (features["assistant_queries_14d"] > 0).astype(int)
features.shape

(24608, 13)

In [7]:
numeric = ["sessions_14d", "searches_14d", "clicks_14d", "purchases_14d", "any_purchase_14d", "used_assistant_14d"]
X = pd.get_dummies(features[numeric + ["platform", "acquisition_channel", "country"]],
                   columns=["platform", "acquisition_channel", "country"], drop_first=True, dtype=float)
y = features["retained_d90"].astype(int)

train = features["signup_date"] < pd.Timestamp("2026-02-01")
test = (features["signup_date"] >= pd.Timestamp("2026-02-01")) & (features["signup_date"] < pd.Timestamp("2026-08-01"))

scaler = StandardScaler().fit(X[train])
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X[train]), y[train])

p_test = model.predict_proba(scaler.transform(X[test]))[:, 1]
auc = roc_auc_score(y[test], p_test)
brier = brier_score_loss(y[test], p_test)
print(f"train users: {train.sum():,}  test users: {test.sum():,}")
print(f"hold-out AUC = {auc:.3f}   Brier score = {brier:.3f}   base rate = {y[test].mean():.3f}")

train users: 13,989  test users: 10,512
hold-out AUC = 0.754   Brier score = 0.194   base rate = 0.382


In [8]:
# Calibration: split test users into 10 groups by predicted probability, compare with what happened
cal = pd.DataFrame({"p": p_test, "y": y[test].to_numpy()})
cal["decile"] = pd.qcut(cal["p"], 10, labels=False, duplicates="drop")
calib = cal.groupby("decile").agg(predicted=("p", "mean"), observed=("y", "mean"), users=("y", "size")).round(3)
calib.to_csv(OUT / "nb_retention_calibration.csv")
calib

,predicted,observed,users
decile,,,
0,0.151,0.102,1055
1,0.184,0.124,1055
2,0.213,0.207,1051
3,0.246,0.273,1045
4,0.289,0.320,1052
5,0.348,0.403,1049
6,0.427,0.445,1051
7,0.517,0.546,1051
8,0.637,0.634,1051


In [9]:
# Coefficients as odds ratios for +1 standard deviation (or presence for 0/1 dummies after scaling)
coef = pd.DataFrame({"feature": X.columns, "coef_std": model.coef_[0]})
coef["odds_ratio_per_sd"] = np.exp(coef["coef_std"]).round(2)
coef = coef.sort_values("coef_std", ascending=False)
coef.to_csv(OUT / "nb_retention_coefficients.csv", index=False)

# Plain-language effect of a first purchase: compare the observed retention of users with / without one
first_purchase_effect = features.groupby("any_purchase_14d")["retained_d90"].mean().round(3).to_dict()
json.dump({"auc": round(auc, 3), "brier": round(brier, 3), "base_rate": round(float(y[test].mean()), 3),
           "train_users": int(train.sum()), "test_users": int(test.sum()),
           "retention_by_first_purchase_14d": first_purchase_effect, "rates": summary_rates},
          open(OUT / "nb_summary.json", "w"), indent=1)
coef

,feature,coef_std,odds_ratio_per_sd
0,sessions_14d,0.743610,2.10
4,any_purchase_14d,0.454810,1.58
1,searches_14d,0.138727,1.15
5,used_assistant_14d,0.054545,1.06
12,country_US,0.026690,1.03
3,purchases_14d,0.018146,1.02
6,platform_extension,0.015534,1.02
11,acquisition_channel_referral,0.013069,1.01
9,acquisition_channel_organic,0.001408,1.00
7,platform_ios,-0.019029,0.98


**Reading the coefficients.** An odds ratio of 1.5 means: one standard deviation more of this behaviour multiplies the odds of being active at day 90 by 1.5,
*holding the other features constant*. These are associations in simulated data, not causes: users who buy early are also more motivated users.
In real life the model would be used to **rank** new users for onboarding actions, and any action would be tested with an A/B test.

## 4. Does the AI assistant increase purchases? (SIMULATED)

The simulation contains a **known** true effect: using the assistant raises weekly purchase frequency by **+10%**.
But engaged users are more likely to adopt it (selection bias). Compare:
- the **naive** comparison (adopters vs non-adopters);
- an **adjusted** comparison that controls for engagement measured *before* adoption (first-week sessions).

In [10]:
act = con.execute('''
    WITH adopters AS (
        SELECT user_id, min(event_date) AS first_assistant_date
        FROM events WHERE event_type = 'assistant_query' GROUP BY user_id
    ),
    week1 AS (
        SELECT e.user_id, sum(event_count) AS sessions_week1
        FROM events e JOIN users u USING (user_id)
        WHERE event_type = 'session' AND e.event_date < u.signup_date + INTERVAL 7 DAY
        GROUP BY e.user_id
    ),
    life AS (
        SELECT u.user_id,
               greatest(1, date_diff('week', u.signup_date,
                        least(DATE '2026-10-31', coalesce(max(e.event_date), u.signup_date)))) AS weeks_active
        FROM users u LEFT JOIN events e USING (user_id) GROUP BY u.user_id, u.signup_date
    )
    SELECT u.user_id, (a.user_id IS NOT NULL)::INT AS adopter, coalesce(w.sessions_week1, 0) AS sessions_week1,
           l.weeks_active, coalesce(p.n, 0) AS purchases
    FROM users u
    LEFT JOIN adopters a USING (user_id)
    LEFT JOIN week1 w USING (user_id)
    LEFT JOIN life l USING (user_id)
    LEFT JOIN (SELECT user_id, count(*) AS n FROM purchases GROUP BY user_id) p USING (user_id)
''').fetchdf()
act["purchases_per_week"] = act["purchases"] / act["weeks_active"]
naive = act.groupby("adopter")["purchases_per_week"].mean()
naive_uplift = naive[1] / naive[0] - 1

# Adjusted: Poisson regression of purchases with log(weeks) offset, controlling for week-1 sessions
import statsmodels.api as sm
Xa = sm.add_constant(pd.DataFrame({"adopter": act["adopter"], "log_sessions_week1": np.log1p(act["sessions_week1"])}))
pois = sm.GLM(act["purchases"], Xa, family=sm.families.Poisson(), offset=np.log(act["weeks_active"])).fit()
adj_uplift = np.exp(pois.params["adopter"]) - 1
ci = np.exp(pois.conf_int().loc["adopter"]) - 1
assistant = {"naive_uplift_pct": round(naive_uplift * 100, 1), "adjusted_uplift_pct": round(adj_uplift * 100, 1),
             "adjusted_ci95_pct": [round(ci[0] * 100, 1), round(ci[1] * 100, 1)], "true_simulated_uplift_pct": 10.0,
             "adoption_rate_pct": round(act["adopter"].mean() * 100, 1)}
json.dump(assistant, open(OUT / "nb_assistant_effect.json", "w"), indent=1)
assistant

{'naive_uplift_pct': np.float64(86.0),
 'adjusted_uplift_pct': np.float64(32.9),
 'adjusted_ci95_pct': [np.float64(30.9), np.float64(35.0)],
 'true_simulated_uplift_pct': 10.0,
 'adoption_rate_pct': np.float64(22.7)}

**Reading.** The naive comparison mixes the assistant's effect with the fact that adopters were already more active.
Controlling for pre-adoption engagement moves the estimate much closer to the true +10%, but only an **A/B test** (random access to the assistant,
or a randomised rollout) would give a clean causal answer on real data. This is exactly the question a Data team gets about Juno.